# Concept LoRA — generate → YOLO-segment → paste into original (100% bg preserved)

**Inference-only.** Loads a trained **concept** LoRA (text→image) and, for each
original background image:

1. the concept model **generates** a full image containing a person (from the
   prompt only — it does NOT see the original).
2. **YOLOv8-seg** segments that generated person out (a clean, tight cut —
   `retina_masks` for crisp edges + a small erode to drop the background halo).
3. the person is **pasted onto the ORIGINAL** background, feathered + colour-matched.

Net effect: the original background is preserved **byte-exact outside the person**
(composite, not hard-restore). Set `FEATHER_PX = 0` for a strictly byte-exact
background (no blended seam band).

> The concept model is blind to the original, so the person's placement/scale
> come from the generation — steer it with the prompt.

Needs GPU + SD3.5 access + a trained **concept** adapter (mount your run as a dataset).

## 1. Setup

In [ ]:
import subprocess, sys, os
from pathlib import Path
REPO = Path('/kaggle/working/VIN')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/BDT-17/VIN.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin'], check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'], check=True)
sys.path.insert(0, str(REPO))
print('repo at', subprocess.run(['git','-C',str(REPO),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())
subprocess.run([sys.executable,'-m','pip','install','-q','--force-reinstall','--no-deps',
                'transformers==4.46.3','tokenizers==0.20.3','huggingface_hub==0.25.2'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q',
                'diffusers==0.31.0','accelerate==0.34.2','peft==0.13.2',
                'safetensors>=0.4.3','sentencepiece','protobuf','pillow>=10','numpy'], check=True)
# ultralytics (YOLOv8-seg) does the person segmentation; --no-deps keeps the pins above.
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','ultralytics'], check=True)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import transformers, diffusers, torch
assert transformers.__version__ == '4.46.3'
from transformers.utils import FLAX_WEIGHTS_NAME
assert torch.cuda.is_available(); print('OK on', torch.cuda.get_device_name(0))

## 2. SD3.5 access + locate the trained concept adapter

Set `ADAPTER_DIR` to the folder that **directly contains**
`pytorch_lora_weights.safetensors`:

- finished run → `<run>/adapter`
- **interrupted run** → the latest checkpoint, e.g. `<run>/checkpoints/checkpoint-2500`
- uploaded the checkpoint folder as a Kaggle dataset → `/kaggle/input/<your-dataset>`

Leave it `None` to auto-detect (prefers a finished `adapter/`, otherwise the
highest-numbered `checkpoint-N`). A mid-train checkpoint has no
`training_provenance.json` yet — that is fine, it is loaded as a concept LoRA.

In [ ]:
import json
from pathlib import Path
_local = Path('/kaggle/input/stable-diffusion-3-5-medium')
HF_TOKEN = None
if _local.exists():
    SD35_MODEL = str(_local)
else:
    SD35_MODEL = 'stabilityai/stable-diffusion-3.5-medium'
    try:
        from kaggle_secrets import UserSecretsClient; HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        import os; HF_TOKEN = os.environ.get('HF_TOKEN')
    assert HF_TOKEN, 'Need HF_TOKEN or mounted SD3.5'
    from huggingface_hub import login; login(token=HF_TOKEN)

# >>> ADAPTER_DIR: the folder that DIRECTLY contains pytorch_lora_weights.safetensors.
# For a finished run that is <run>/adapter. For an INTERRUPTED run use the latest
# checkpoint, e.g. <run>/checkpoints/checkpoint-2500. If you uploaded just the
# checkpoint folder (or the bare .safetensors) as a Kaggle dataset, point at
# /kaggle/input/<your-dataset>. Leave None to auto-detect any
# pytorch_lora_weights.safetensors under the roots below (prefers adapter/, else
# the highest-numbered checkpoint-N).
ADAPTER_DIR = None

LORA_FILE = 'pytorch_lora_weights.safetensors'

def _is_edit_run(folder):
    """True only if this is the mask-free EDIT flow (needs input_proj). A mid-train
    checkpoint has no provenance -> treated as concept (the common case here)."""
    for pv in [folder / 'training_provenance.json',
               folder.parent / 'training_provenance.json',
               folder.parent.parent / 'training_provenance.json']:
        try:
            if pv.exists():
                return bool(json.loads(pv.read_text(encoding='utf-8')).get('requires_input_proj', False))
        except Exception:
            pass
    return False

def _ckpt_step(folder):
    import re
    m = re.search(r'checkpoint-(\d+)', folder.name)
    return int(m.group(1)) if m else -1

if ADAPTER_DIR is None:
    roots = [Path('/kaggle/input'), Path('/kaggle/working/vin_lora/models')]
    cands = []
    for root in roots:
        if root.exists():
            cands += [p.parent for p in root.rglob(LORA_FILE)]
    cands = [c for c in cands if not _is_edit_run(c)]
    adapters = [c for c in cands if c.name == 'adapter']           # finished run
    ADAPTER_DIR = adapters[0] if adapters else (max(cands, key=_ckpt_step) if cands else None)
else:
    ADAPTER_DIR = Path(ADAPTER_DIR)

assert ADAPTER_DIR is not None and (ADAPTER_DIR / LORA_FILE).exists(), (
    f'No {LORA_FILE} found. Set ADAPTER_DIR to the folder that contains it — a '
    'finished <run>/adapter, an interrupted <run>/checkpoints/checkpoint-N, or a '
    'mounted Kaggle dataset folder.')
assert not _is_edit_run(ADAPTER_DIR), (
    'This adapter is the mask-free EDIT flow (requires input_proj), not a concept LoRA.')
print('concept adapter dir:', ADAPTER_DIR)

## 3. Load the concept runner + the YOLOv8-seg segmenter (once)

In [ ]:
from LoRA.inference.sd35_concept_runner import SD35ConceptRunner
from LoRA.inference.person_detector import load_person_segmenter
# Load straight from ADAPTER_DIR (works for a finished adapter/ OR a mid-train
# checkpoint-N folder). We bypass load_concept_runner_from_run, which requires a
# training_provenance.json that an interrupted run has not written yet.
runner = SD35ConceptRunner(SD35_MODEL, ADAPTER_DIR, hf_token=HF_TOKEN).load()
# A larger seg model + retina_masks => crisper, cleaner person silhouette ('cắt rõ').
# yolov8x-seg = best masks; switch to yolov8m-seg / yolov8s-seg for faster CPU inference.
SEG_WEIGHTS = 'yolov8x-seg.pt'
segmenter = load_person_segmenter(SEG_WEIGHTS, device='cpu')
print('concept runner + segmenter ready | adapter =', ADAPTER_DIR, '| seg =', SEG_WEIGHTS)

## 4. Background images (from a dataset in sources.yaml) + settings

`BG_SOURCE` = `citypersons` | `mot17_02` | `human_detection`. Pulls real images
straight from the mounted dataset; falls back to `/kaggle/working/my_backgrounds`.

In [ ]:
from pathlib import Path
from LoRA.data.list_images import list_source_images

BG_SOURCE = 'citypersons'   # 'citypersons' | 'mot17_02' | 'human_detection'
N_BG = 6                    # how many images to process
STEPS = 28
GUIDANCE = 7.0             # text-to-image CFG scale
FEATHER_PX = 2             # seam softness; set 0 for a strictly byte-exact background
ERODE_PX = 1               # shrink mask inward to drop YOLO bg halo ('cắt chặt'); 0 = off
COLOR_MATCH = 0.5          # 0..1 colour transfer toward the original (anti-sticker)
POISSON = False            # True = OpenCV seamlessClone for the largest person
# Concept captions were trained as 'a photo of <subject>' — match that style.
PROMPT = 'a photo of a person walking on a city street'
SOURCES_YAML = Path('/kaggle/working/VIN/LoRA/configs/sources.yaml')

try:
    bg_paths = list_source_images(BG_SOURCE, limit=N_BG, sources_path=SOURCES_YAML)
    print(f'{len(bg_paths)} images from dataset source "{BG_SOURCE}"')
except FileNotFoundError as e:
    print('dataset not mounted:', e)
    BG_DIR = Path('/kaggle/working/my_backgrounds'); BG_DIR.mkdir(parents=True, exist_ok=True)
    bg_paths = sorted([p for p in BG_DIR.glob('*') if p.suffix.lower() in ('.png','.jpg','.jpeg')])[:N_BG]
    print(f'{len(bg_paths)} images from {BG_DIR}')
assert bg_paths, f'No images for "{BG_SOURCE}" (mount via Add Data) or in the fallback folder.'

## 5. Generate → segment → paste, per image

In [ ]:
import time
from PIL import Image
from LoRA.inference.segment_paste import generate_and_paste_concept
OUT = Path('/kaggle/working/concept_segpaste'); (OUT/'images').mkdir(parents=True, exist_ok=True)
results = []
for i, bp in enumerate(bg_paths):
    orig = Image.open(bp).convert('RGB')
    te = time.time()
    composite, generated, info = generate_and_paste_concept(
        runner, orig, PROMPT, segmenter,
        seed=42 + i, num_inference_steps=STEPS, guidance_scale=GUIDANCE,
        feather_px=FEATHER_PX, erode_px=ERODE_PX, color_match=COLOR_MATCH, poisson=POISSON)
    composite.save(OUT/'images'/f'{bp.stem}_segpaste.png')
    results.append((bp.stem, orig, generated, composite, info))
    print(f'[{i+1}/{len(bg_paths)}] {bp.name}  {time.time()-te:.1f}s  pasted={info["pasted"]} confs={info.get("confs")}', flush=True)
print('outputs ->', OUT)

## 6. Show: original | generated (full) | composite (person pasted onto original)

In [ ]:
from PIL import Image
from IPython.display import display
for name, orig, generated, composite, info in results:
    cells = [orig.resize((256,256)), generated.resize((256,256)), composite.resize((256,256))]
    strip = Image.new('RGB', (768, 256))
    for j, im in enumerate(cells):
        strip.paste(im, (256*j, 0))
    print(f'{name}  (original | generated | composite)  pasted={info["pasted"]}')
    display(strip)